# Sports Card Extractor (Enhanced)

Extract structured data from sports card images using OpenAI's vision API with **dual-image support** (front + back).

## Features
- 📁 **Local file upload** - Upload images from your computer
- 🔄 **Front + Back processing** - Extract from both sides simultaneously
- 🎯 **Comprehensive extraction** - Brand, year, player, card #, serial, auto, parallels, visual descriptors
- 🔍 **eBay search generation** - Ready-to-use search URLs

## 1. Setup & Install

In [ ]:
!pip install openai python-dotenv pydantic pandas ipywidgets --quiet

In [ ]:
import os
import base64
import webbrowser
from pathlib import Path
from urllib.parse import quote
from typing import Optional, List

import openai
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from IPython.display import display, Image, Markdown, HTML
import ipywidgets as widgets
from IPython.display import clear_output

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

## 2. Data Model (Pydantic)

Comprehensive schema for sports cards based on industry metadata standards.

In [ ]:
class SportsCard(BaseModel):
    """Comprehensive structured representation of a sports card."""

    # === CORE IDENTIFICATION ===
    player_name: str = Field(
        description="Full name of the player (e.g. 'Aaron Judge', 'Patrick Mahomes II')"
    )
    team: Optional[str] = Field(
        default=None,
        description="Team name shown on card (e.g. 'New York Yankees', 'Kansas City Chiefs')"
    )
    position: Optional[str] = Field(
        default=None,
        description="Player position (e.g. 'OF', 'QB', 'PG', 'C', 'WR')"
    )
    sport: Optional[str] = Field(
        default=None,
        description="Sport: Baseball, Football, Basketball, Hockey, Soccer"
    )
    
    # === CARD DETAILS (often on back) ===
    manufacturer: Optional[str] = Field(
        default=None,
        description="Card manufacturer/brand (e.g. 'Topps', 'Panini', 'Upper Deck', 'Bowman', 'Donruss', 'Fleer')"
    )
    product_set: Optional[str] = Field(
        default=None,
        description="Product/Set name (e.g. 'Topps Chrome', 'Prizm', 'Finest', 'Optic', 'Mosaic', 'Select')"
    )
    year: Optional[int] = Field(
        default=None,
        description="Year from copyright/fine print on back of card (e.g. 2024, 2025)"
    )
    card_number: Optional[str] = Field(
        default=None,
        description="Card number from back of card (e.g. '51', '150', 'RC-25')"
    )
    
    # === RARITY & VARIANTS ===
    serial_number: Optional[str] = Field(
        default=None,
        description="Serial numbering if present (e.g. '25/99', '1/1', '517/2000'). Found on front or back."
    )
    parallel_type: Optional[str] = Field(
        default=None,
        description="Parallel/refractor type (e.g. 'Silver Prizm', 'Gold Refractor', 'Blue Wave', 'Cracked Ice', 'Superfractor', 'X-Fractor', 'Aqua Lava', 'Snakeskin')"
    )
    insert_name: Optional[str] = Field(
        default=None,
        description="Insert set name if not a base card (e.g. 'League Leaders', 'Award Winners', 'Rookie Debut')"
    )
    
    # === SPECIAL FEATURES ===
    is_rookie_card: Optional[bool] = Field(
        default=None,
        description="True if marked as rookie card (RC logo, 'Rookie' text, or 1st Bowman)"
    )
    is_autograph: Optional[bool] = Field(
        default=None,
        description="True if card contains an autograph (usually on front)"
    )
    is_memorabilia: Optional[bool] = Field(
        default=None,
        description="True if card contains jersey patch, relic, or game-used material"
    )
    
    # === GRADING (if slabbed) ===
    grading_company: Optional[str] = Field(
        default=None,
        description="Grading company if card is slabbed (e.g. 'PSA', 'BGS', 'SGC', 'CGC')"
    )
    grade: Optional[str] = Field(
        default=None,
        description="Grade value (e.g. '10', '9.5', 'GEM MT 10', 'PRISTINE 10')"
    )
    
    # === VISUAL DESCRIPTION (for comprehensive search) ===
    visual_description: Optional[str] = Field(
        default=None,
        description="Visual characteristics: colors, patterns, finishes, special effects (e.g. 'purple border, holographic shimmer, stained glass pattern, rainbow refractor finish, cracked ice texture')"
    )
    
    # === ADDITIONAL IDENTIFIERS ===
    additional_text: Optional[str] = Field(
        default=None,
        description="Any other identifying text on the card (e.g. 'League Leaders', 'All-Star', 'MVP', 'Photo Variation', 'Short Print SP', 'Super Short Print SSP')"
    )

    @property
    def display_name(self) -> str:
        parts = []
        if self.year:
            parts.append(str(self.year))
        if self.manufacturer:
            parts.append(self.manufacturer)
        if self.product_set:
            parts.append(self.product_set)
        parts.append(self.player_name)
        if self.parallel_type:
            parts.append(f"- {self.parallel_type}")
        if self.is_rookie_card:
            parts.append("RC")
        if self.serial_number:
            parts.append(f"[{self.serial_number}]")
        return " ".join(parts)

    @property
    def ebay_search_query(self) -> str:
        """Build optimized eBay search string."""
        tokens = []
        if self.year:
            tokens.append(str(self.year))
        if self.manufacturer:
            tokens.append(self.manufacturer)
        if self.product_set:
            tokens.append(self.product_set)
        tokens.append(self.player_name)
        if self.card_number:
            tokens.append(f"#{self.card_number}")
        if self.parallel_type:
            tokens.append(self.parallel_type)
        if self.serial_number:
            parts = self.serial_number.split('/')
            if len(parts) == 2:
                tokens.append(f"/{parts[1]}")
        if self.is_rookie_card:
            tokens.append("RC")
        if self.is_autograph:
            tokens.append("Auto")
        if self.is_memorabilia:
            tokens.append("Relic")
        if self.grade:
            company = self.grading_company or "PSA"
            tokens.append(f"{company} {self.grade}")
        return " ".join(tokens)

    @property
    def ebay_url(self) -> str:
        return f"https://www.ebay.com/sch/i.html?_nkw={quote(self.ebay_search_query)}"

    def to_markdown_table(self) -> str:
        lines = ["| **Field** | **Value** |", "| :-- | :-- |"]
        for field_name, value in self.model_dump().items():
            if value is not None and value != [] and value != False:
                label = field_name.replace("_", " ").title()
                if isinstance(value, bool):
                    value = "✅ Yes"
                if isinstance(value, list):
                    value = ", ".join(str(v) for v in value)
                lines.append(f"| {label} | {value} |")
        return "\n".join(lines)

    def _repr_markdown_(self):
        return f"### {self.display_name}\n\n{self.to_markdown_table()}"

## 3. Image Encoding

In [ ]:
def encode_uploaded_file(file_info: dict) -> dict:
    """Convert uploaded file content to API-ready format."""
    content = file_info['content']
    name = file_info.get('name', 'image.jpg')
    ext = os.path.splitext(name)[1].lower().lstrip(".")
    mime = {"jpg": "jpeg", "jpeg": "jpeg", "png": "png", "webp": "webp", "gif": "gif"}.get(ext, "jpeg")
    
    # Handle bytes or memoryview
    if isinstance(content, memoryview):
        content = bytes(content)
    
    b64 = base64.b64encode(content).decode("utf-8")
    return {
        "type": "image_url",
        "image_url": {"url": f"data:image/{mime};base64,{b64}"},
    }

## 4. Extraction Function (Dual Image Support)

In [ ]:
SYSTEM_PROMPT = """
You are an expert sports card analyst and collector. You will receive images of a sports card 
(front and/or back). Extract all structured information following these priorities:

EXTRACTION PRIORITY ORDER:
1. BRAND/MANUFACTURER - Look for Topps, Panini, Upper Deck, Bowman, Donruss, Fleer logos
2. PRODUCT SET - Identify the specific product (Chrome, Prizm, Finest, Optic, Mosaic, Select, etc.)

3. YEAR - **CRITICAL INSTRUCTION**:
   ⚠️ ONLY extract year from the COPYRIGHT LINE in the fine print (e.g., "© 2025 Topps", "TM & © 2025")
   ⚠️ DO NOT use years from statistics, performance data, or season records
   ⚠️ Stats showing "2024 SEASON" means the CARD is from 2025 (stats are always from previous year)
   ⚠️ Look for: "© YYYY", "TM & © YYYY", or copyright symbol followed by year
   
4. PLAYER NAME - Full name as shown on card
5. CARD NUMBER - Usually on BACK, sometimes front (e.g., "#51", "RC-25")
6. SERIAL NUMBER - Limited print notation like "25/99" or "1/1" (front or back)
7. AUTOGRAPH - Look for signature, usually on FRONT
8. PARALLEL TYPE - Identify color variants, refractors, special finishes:
   - Topps: Refractor, Gold, Blue, Green, Purple, Rainbow Foil, X-Fractor, Superfractor
   - Panini: Silver Prizm, Blue Prizm, Green Wave, Cracked Ice, Snakeskin, Disco, Lazer
   - Upper Deck: Exclusives, Clear Cut, High Gloss, Outburst
9. VISUAL DESCRIPTION - Describe colors, patterns, special effects you observe
10. OTHER IDENTIFIERS - Insert names, SP/SSP, Photo Variation, Award mentions

IMPORTANT:
- Use null for any field you cannot confidently determine
- The BACK of the card is crucial for: Year (COPYRIGHT ONLY), Card Number, Copyright info
- The FRONT is crucial for: Player image, Autograph, Parallel appearance, Serial number
- Be precise with serial numbers (format: XX/YY)
- Visual description should help with search (colors, effects, textures)
- REMEMBER: Card year = Copyright year, NOT stats year
"""


def extract_card(front_file: dict = None, back_file: dict = None) -> SportsCard:
    """
    Extract card data from uploaded front and/or back images.
    
    Args:
        front_file: Dict with 'content' and 'name' from FileUpload widget
        back_file: Dict with 'content' and 'name' from FileUpload widget
    """
    content_parts = [{"type": "text", "text": "Extract all structured data from this sports card. I'm providing the front and/or back images."}]
    
    if front_file:
        content_parts.append({"type": "text", "text": "FRONT OF CARD:"})
        content_parts.append(encode_uploaded_file(front_file))
    
    if back_file:
        content_parts.append({"type": "text", "text": "BACK OF CARD:"})
        content_parts.append(encode_uploaded_file(back_file))
    
    if len(content_parts) == 1:
        raise ValueError("Must provide at least one image (front or back)")
    
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": content_parts},
        ],
        response_format=SportsCard,
        temperature=0.1,
    )

    card = completion.choices[0].message.parsed

    if card is None:
        refusal = completion.choices[0].message.refusal
        raise ValueError(f"Extraction failed: {refusal or 'No structured output returned'}")

    return card

## 5. Upload Your Card Images

Upload front and/or back images of your sports card.

In [ ]:
# Create upload widgets
front_upload = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Front of Card'
)

back_upload = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Back of Card'
)

extract_button = widgets.Button(
    description='Extract Card Data',
    button_style='success',
    icon='search'
)

output = widgets.Output()

# Store the extracted card for later use
extracted_card = None

def on_extract_click(b):
    global extracted_card
    with output:
        clear_output()
        
        front_data = None
        back_data = None
        
        # FileUpload.value is a tuple of dicts in newer ipywidgets
        if front_upload.value:
            # Handle both old dict format and new tuple format
            if isinstance(front_upload.value, tuple):
                front_data = front_upload.value[0]
            elif isinstance(front_upload.value, dict):
                front_data = list(front_upload.value.values())[0]
            print(f"✓ Front image loaded")
        
        if back_upload.value:
            if isinstance(back_upload.value, tuple):
                back_data = back_upload.value[0]
            elif isinstance(back_upload.value, dict):
                back_data = list(back_upload.value.values())[0]
            print(f"✓ Back image loaded")
        
        if not front_data and not back_data:
            print("⚠️ Please upload at least one image (front or back)")
            return
        
        print("\n🔍 Extracting card data...\n")
        
        try:
            extracted_card = extract_card(front_file=front_data, back_file=back_data)
            display(extracted_card)
            print(f"\n📝 eBay Search Query: {extracted_card.ebay_search_query}")
            print(f"\n🔗 eBay URL: {extracted_card.ebay_url}")
        except Exception as e:
            print(f"❌ Error: {e}")
            import traceback
            traceback.print_exc()

extract_button.on_click(on_extract_click)

print("📤 Upload your card images:")
print("   - Front of card (recommended for parallel/auto/visual)")
print("   - Back of card (recommended for year, card number, fine print)")
print("")
display(widgets.VBox([
    widgets.HBox([widgets.Label("Front:"), front_upload]),
    widgets.HBox([widgets.Label("Back: "), back_upload]),
    extract_button,
    output
]))

## 6. View Extracted Data

In [ ]:
# Access the extracted card data (after running extraction above)
if extracted_card:
    print("=== EXTRACTED DATA ===")
    print(f"Player: {extracted_card.player_name}")
    print(f"Team: {extracted_card.team}")
    print(f"Sport: {extracted_card.sport}")
    print(f"Year: {extracted_card.year}")
    print(f"Brand: {extracted_card.manufacturer}")
    print(f"Set: {extracted_card.product_set}")
    print(f"Card #: {extracted_card.card_number}")
    print(f"Serial: {extracted_card.serial_number}")
    print(f"Parallel: {extracted_card.parallel_type}")
    print(f"Rookie: {extracted_card.is_rookie_card}")
    print(f"Auto: {extracted_card.is_autograph}")
    print(f"Relic: {extracted_card.is_memorabilia}")
    if extracted_card.grade:
        print(f"Grade: {extracted_card.grading_company} {extracted_card.grade}")
    print(f"\nVisual: {extracted_card.visual_description}")
    print(f"Other: {extracted_card.additional_text}")
else:
    print("No card extracted yet. Upload images and click 'Extract Card Data' above.")

In [ ]:
# Raw JSON output
if extracted_card:
    print(extracted_card.model_dump_json(indent=2))

## 7. Open eBay Search

In [ ]:
# Open eBay search in browser
if extracted_card:
    print(f"Search query: {extracted_card.ebay_search_query}")
    display(Markdown(f"**[🔍 Search eBay for this card]({extracted_card.ebay_url})**"))
    
    # Uncomment to auto-open in browser:
    # webbrowser.open(extracted_card.ebay_url)

## 8. Tips for Best Results

### Image Quality
- Use well-lit, clear photos
- Capture the entire card (no cropping)
- For graded cards, capture inside the slab

### Front + Back
- **Front**: Parallel type, autograph, serial number, visual appearance
- **Back**: Year (copyright), card number, fine print details

### Parallel Identification
The model knows major parallel types:
- **Topps**: Refractor, Gold, Blue, Green, Purple, Rainbow Foil, X-Fractor, Superfractor
- **Panini**: Silver Prizm, Blue Prizm, Green Wave, Cracked Ice, Snakeskin, Disco, Lazer
- **Upper Deck**: Exclusives, Clear Cut, High Gloss, Outburst

### Visual Description
The `visual_description` field captures colors and effects to help with search:
- "Purple border, holographic shimmer"
- "Cracked ice pattern, rainbow refractor"
- "Stained glass appearance, gold trim"